# LeetCode 1522 — Diameter of N-Ary Tree bằng Python

Link đề: https://leetcode.com/problems/diameter-of-n-ary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, DP on Trees

> **Ghi chú Premium:** đây là bài Premium 🔒. Đề bài và chữ ký trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1522/). Nếu có tài khoản Premium, bạn nên đối chiếu lại đề gốc.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children if children is not None else []
"""

class Solution:
    def diameter(self, root: 'Node') -> int:
        """
        :type root: 'Node'
        :rtype: int
        """
        ...
```

## 1. Mục tiêu học

1. Hiểu **đường kính** của cây: độ dài (số **cạnh**) của đường đi dài nhất giữa hai nút bất kỳ — không nhất thiết đi qua gốc.
2. Dựng cây N-phân từ dạng level-order có `null` phân nhóm (như LeetCode) để tự test.
3. Viết được 4 cách: brute force BFS từ mọi nút `O(n²)`, **BFS hai lần** trên đồ thị vô hướng `O(n)`, **DFS hậu tố trả chiều cao + giữ top-2** (lời giải doocs), và bản **lặp** không đệ quy.
4. Nắm kỹ thuật cập nhật **hai giá trị lớn nhất** `m1 ≥ m2` trong một lượt duyệt.
5. Phân biệt rõ quy ước "chiều cao tính bằng cạnh hay bằng nút" để tránh lệch 1.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây N-phân, tính **độ dài đường kính** của cây: độ dài đường đi dài nhất giữa **hai nút bất kỳ** (đếm số cạnh). Đường này có thể đi qua hoặc không đi qua gốc.

(Cây được tuần tự hoá theo level-order, mỗi nhóm con ngăn bởi `null`.)

**Ví dụ 1:** `root = [1,null,3,2,4,null,5,6]` → `3` (ví dụ đường 5–3–1–2).

**Ví dụ 2:** `root = [1,null,2,null,3,4,null,5,null,6]` → `4` (đường 5–3–2–4–6, **không** qua gốc 1).

**Ví dụ 3:** `root = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]` → `7`.

**Ràng buộc:** độ sâu cây `≤ 1000`; tổng số nút trong `[1, 10⁴]`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children if children is not None else []
    def __repr__(self):
        return f"Node({self.val})"

ex1 = [1, None, 3, 2, 4, None, 5, 6]
ex2 = [1, None, 2, None, 3, 4, None, 5, None, 6]
ex3 = [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None,
       11, None, 12, None, 13, None, None, 14]
examples = [(ex1, 3), (ex2, 4), (ex3, 7)]
for data, want in examples:
    print(data, "->", want)

[1, None, 3, 2, 4, None, 5, 6] -> 3
[1, None, 2, None, 3, 4, None, 5, None, 6] -> 4
[1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14] -> 7


Hàm phụ tự dựng để test: `build_nary` (level-order → cây), `serialize` (ngược lại), `to_graph` (cây → danh sách kề vô hướng theo giá trị — giá trị trong test là duy nhất).

In [2]:
def build_nary(data):
    if not data:
        return None
    root = Node(data[0])
    q = deque([root])
    i = 2
    while q and i <= len(data):
        cur = q.popleft()
        while i < len(data) and data[i] is not None:
            ch = Node(data[i])
            cur.children.append(ch)
            q.append(ch)
            i += 1
        i += 1
    return root

def serialize(root):
    if root is None:
        return []
    out = [root.val, None]
    q = deque([root])
    while q:
        cur = q.popleft()
        for ch in cur.children:
            out.append(ch.val)
            q.append(ch)
        out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def to_graph(root):
    g = {}
    stack = [root]
    g[root] = []
    while stack:
        u = stack.pop()
        for ch in u.children:
            g[u].append(ch)
            g.setdefault(ch, []).append(u)
            stack.append(ch)
    return g

for data, _ in examples:
    r = build_nary(data)
    print(serialize(r) == data, "| số nút:", len(to_graph(r)))

True | số nút: 6
True | số nút: 6
True | số nút: 14


## 3. Kiến thức nền

### 3.1 Cây N-phân

Mỗi nút có `val` và `children` (list). Lá là nút có `children == []`.

In [3]:
r = build_nary(ex1)
print(r, r.children, [c.children for c in r.children])
def show(node, depth=0):
    print("  " * depth + str(node.val))
    for c in node.children:
        show(c, depth + 1)
show(build_nary(ex2))

Node(1) [Node(3), Node(2), Node(4)] [[Node(5), Node(6)], [], []]
1
  2
    3
      5
    4
      6


### 3.2 Chiều cao (tính bằng cạnh)

`height(lá) = 0`, `height(u) = 1 + max(height(con))`. Lời giải doocs dùng hàm trả `1 + m1` với `m1` = chiều cao con lớn nhất **tính theo số nút trên nhánh** — hai quy ước khác nhau 1; quan trọng là dùng **nhất quán** (xem mục 13).

In [4]:
def height_edges(u):
    return 0 if not u.children else 1 + max(height_edges(c) for c in u.children)

for data, _ in examples:
    print(data[:6], "... chiều cao (cạnh) =", height_edges(build_nary(data)))

[1, None, 3, 2, 4, None] ... chiều cao (cạnh) = 2
[1, None, 2, None, 3, 4] ... chiều cao (cạnh) = 3
[1, None, 2, 3, 4, 5] ... chiều cao (cạnh) = 4


### 3.3 BFS tính khoảng cách trên đồ thị

Từ một nút nguồn, BFS cho khoảng cách (số cạnh) tới mọi nút; nút xa nhất là nút được thăm cuối cùng.

In [5]:
def bfs_far(g, src):
    dist = {src: 0}
    q = deque([src])
    last = src
    while q:
        u = q.popleft()
        last = u
        for v in g[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return last, dist[last]

g = to_graph(build_nary(ex1))
root1 = next(iter(g))
print("xa gốc nhất:", bfs_far(g, root1))

xa gốc nhất: (Node(6), 2)


### 3.4 Giữ hai giá trị lớn nhất trong một lượt

Khi gặp `t`: nếu `t > m1` thì **đẩy** `m1` xuống `m2` rồi `m1 = t`; ngược lại nếu `t > m2` thì `m2 = t`.

In [6]:
m1 = m2 = 0
for t in [3, 1, 4, 1, 5, 9, 2, 6]:
    if t > m1:
        m2, m1 = m1, t
    elif t > m2:
        m2 = t
    print(f"t={t}: m1={m1}, m2={m2}")

t=3: m1=3, m2=0
t=1: m1=3, m2=1
t=4: m1=4, m2=3
t=1: m1=4, m2=3
t=5: m1=5, m2=4
t=9: m1=9, m2=5
t=2: m1=9, m2=5
t=6: m1=9, m2=6


### 3.5 `nonlocal` trong hàm lồng

Hàm DFS lồng bên trong cần **gán** biến `ans` của hàm ngoài → khai báo `nonlocal ans` (chỉ đọc thì không cần).

In [7]:
def outer():
    ans = 0
    def bump(x):
        nonlocal ans
        ans = max(ans, x)
    for x in [3, 7, 2]:
        bump(x)
    return ans
print(outer())

7


## 4. Ý tưởng nền tảng / trực giác

Mọi đường đi trong cây có **một nút cao nhất** (gần gốc nhất) — gọi là "đỉnh" của đường. Từ đỉnh `u`, đường đi xuống **tối đa hai nhánh con khác nhau**. Đường dài nhất có đỉnh `u` = tổng **hai nhánh sâu nhất** đi xuống từ `u`.

Định nghĩa `d(c)` = 1 + chiều cao (cạnh) của con `c` = số cạnh của nhánh dài nhất từ `u` đi qua `c`. Đường dài nhất có đỉnh `u` = `m1 + m2` (hai giá trị `d` lớn nhất).

Tính tay Ví dụ 2 (cây: 1 → 2 → {3, 4}; 3 → 5; 4 → 6):

| nút u | nhánh con d(c) | m1 + m2 | chiều cao u (cạnh) |
|---|---|---|---|
| 5, 6 (lá) | — | 0 | 0 |
| 3 | d(5) = 1 | 1 | 1 |
| 4 | d(6) = 1 | 1 | 1 |
| 2 | d(3) = 2, d(4) = 2 | **4** | 2 |
| 1 | d(2) = 3 | 3 | 3 |

Đáp án = max = **4**, đạt tại đỉnh 2 (không phải gốc).

In [8]:
def manual(u, depth=0):
    ds = []
    for c in u.children:
        h = manual(c, depth + 1)
        ds.append(h + 1)
    ds.sort(reverse=True)
    best_here = sum(ds[:2])
    h = ds[0] if ds else 0
    print("  " * depth + f"nút {u.val}: nhánh={ds} -> đường qua đỉnh này={best_here}, chiều cao={h}")
    return h

_ = manual(build_nary(ex2))

      nút 5: nhánh=[] -> đường qua đỉnh này=0, chiều cao=0
    nút 3: nhánh=[1] -> đường qua đỉnh này=1, chiều cao=1
      nút 6: nhánh=[] -> đường qua đỉnh này=0, chiều cao=0
    nút 4: nhánh=[1] -> đường qua đỉnh này=1, chiều cao=1
  nút 2: nhánh=[2, 2] -> đường qua đỉnh này=4, chiều cao=2
nút 1: nhánh=[3] -> đường qua đỉnh này=3, chiều cao=3


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** đổi cây thành đồ thị vô hướng, chạy BFS từ **mọi** nút, lấy khoảng cách lớn nhất tìm được.

In [9]:
def brute(root):
    g = to_graph(root)
    best = 0
    for src in g:
        _, d = bfs_far(g, src)
        best = max(best, d)
    return best

for data, want in examples:
    print(brute(build_nary(data)), "kỳ vọng", want)

3 kỳ vọng 3
4 kỳ vọng 4
7 kỳ vọng 7


**Độ phức tạp:** `n` lần BFS, mỗi lần `O(n)` → Time `O(n²)` (`10⁸` với `n = 10⁴` — quá chậm trong Python), Space `O(n)`.

**Nút thắt:** BFS từ mọi nút là lãng phí: chỉ cần **hai** lần BFS (định lý về nút xa nhất), hoặc một lần DFS hậu tố.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight A — BFS hai lần:** từ một nút bất kỳ `s`, nút xa nhất `a` luôn là **một đầu của một đường kính**. BFS lần hai từ `a`, khoảng cách xa nhất chính là đường kính.

**Insight B — DP hậu tố (doocs):** mỗi nút trả về chiều cao của nó cho cha; tại mỗi nút, cập nhật `ans = max(ans, m1 + m2)` với `m1, m2` là hai nhánh sâu nhất. Một lượt duyệt xét mọi "đỉnh" có thể.

**Quy tắc:**
- *Nếu* một nút có ≥ 2 con *thì* đường qua nó có thể nối hai nhánh: `m1 + m2`.
- *Nếu* chỉ 1 con *thì* `m2 = 0` → đường từ nút đi xuống.
- Hàm trả về cho cha **chỉ một nhánh** (`m1`), vì đường không thể rẽ hai hướng rồi tiếp tục đi lên.

## 7. Các cách tiếp theo

### Cách 2 — BFS hai lần trên đồ thị vô hướng `O(n)`

1. Dựng đồ thị kề (cha ↔ con).
2. BFS từ gốc → nút xa nhất `a`.
3. BFS từ `a` → khoảng cách xa nhất = đường kính.

In [10]:
def two_bfs(root):
    g = to_graph(root)
    a, _ = bfs_far(g, root)
    b, d = bfs_far(g, a)
    return d

for data, want in examples:
    print(two_bfs(build_nary(data)), "kỳ vọng", want)

3 kỳ vọng 3
4 kỳ vọng 4
7 kỳ vọng 7


In [11]:
class SolutionTwoBFS:
    def diameter(self, root: 'Node') -> int:
        g = {root: []}
        stack = [root]
        while stack:
            u = stack.pop()
            for ch in u.children:
                g[u].append(ch)
                g[ch] = [u]
                stack.append(ch)

        def far(src):
            dist = {src: 0}
            q = deque([src])
            last = src
            while q:
                u = q.popleft()
                last = u
                for v in g[u]:
                    if v not in dist:
                        dist[v] = dist[u] + 1
                        q.append(v)
            return last, dist[last]

        a, _ = far(root)
        return far(a)[1]

print([SolutionTwoBFS().diameter(build_nary(d)) for d, _ in examples])

[3, 4, 7]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (đồ thị + dict khoảng cách).

### Cách 3 — DFS hậu tố trả chiều cao, giữ top-2 (lời giải doocs)

`dfs(u)` trả `1 + m1` = số nút trên nhánh dài nhất từ `u` xuống (tức chiều cao theo cạnh + 1). Với mỗi con `c`, `t = dfs(c)` chính là số cạnh của nhánh từ `u` đi qua `c`. Cập nhật top-2 và `ans = max(ans, m1 + m2)`.

In [12]:
def dfs_top2(root):
    ans = 0
    def dfs(u):
        nonlocal ans
        if u is None:
            return 0
        m1 = m2 = 0
        for c in u.children:
            t = dfs(c)
            if t > m1:
                m2, m1 = m1, t
            elif t > m2:
                m2 = t
        ans = max(ans, m1 + m2)
        return 1 + m1
    dfs(root)
    return ans

for data, want in examples:
    print(dfs_top2(build_nary(data)), "kỳ vọng", want)

3 kỳ vọng 3
4 kỳ vọng 4
7 kỳ vọng 7


In [13]:
class SolutionDFS:
    def diameter(self, root: 'Node') -> int:
        """
        :type root: 'Node'
        :rtype: int
        """

        def dfs(root):
            if root is None:
                return 0
            nonlocal ans
            m1 = m2 = 0
            for child in root.children:
                t = dfs(child)
                if t > m1:
                    m2, m1 = m1, t
                elif t > m2:
                    m2 = t
            ans = max(ans, m1 + m2)
            return 1 + m1

        ans = 0
        dfs(root)
        return ans

print([SolutionDFS().diameter(build_nary(d)) for d, _ in examples])

[3, 4, 7]


**Độ phức tạp:** Time `O(n)`, Space `O(h)` cho đệ quy (`h ≤ 1000`). Với `h` gần 1000, Python có thể chạm giới hạn đệ quy mặc định (~1000) → nên có bản lặp.

### Cách 4 — DFS hậu tố **lặp** (không đệ quy)

1. Lấy thứ tự tiền tố bằng stack (cha trước con), duyệt **ngược** → con trước cha.
2. `h[u]` = `1 + m1` (giống Cách 3); tính từ `h` của các con.

In [14]:
def iterative_top2(root):
    order, stack = [], [root]
    while stack:
        u = stack.pop()
        order.append(u)
        stack.extend(u.children)
    h = {}
    ans = 0
    for u in reversed(order):
        m1 = m2 = 0
        for c in u.children:
            t = h[c]
            if t > m1:
                m2, m1 = m1, t
            elif t > m2:
                m2 = t
        ans = max(ans, m1 + m2)
        h[u] = 1 + m1
    return ans

for data, want in examples:
    print(iterative_top2(build_nary(data)), "kỳ vọng", want)

3 kỳ vọng 3
4 kỳ vọng 4
7 kỳ vọng 7


**Độ phức tạp:** Time `O(n)`, Space `O(n)`. An toàn với mọi độ sâu.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 1 (`1 → {3, 2, 4}`, `3 → {5, 6}`):

| nút | các t từ con | m1 | m2 | ans sau | trả về 1+m1 |
|---|---|---|---|---|---|
| 5 | — | 0 | 0 | 0 | 1 |
| 6 | — | 0 | 0 | 0 | 1 |
| 3 | 1, 1 | 1 | 1 | 2 | 2 |
| 2 | — | 0 | 0 | 2 | 1 |
| 4 | — | 0 | 0 | 2 | 1 |
| 1 | 2, 1, 1 | 2 | 1 | **3** | 3 |

In [15]:
def trace_dfs(root):
    ans = 0
    def dfs(u, depth):
        nonlocal ans
        m1 = m2 = 0
        ts = []
        for c in u.children:
            t = dfs(c, depth + 1)
            ts.append(t)
            if t > m1:
                m2, m1 = m1, t
            elif t > m2:
                m2 = t
        ans = max(ans, m1 + m2)
        print("  " * depth + f"nút {u.val}: t={ts} m1={m1} m2={m2} ans={ans} -> trả {1 + m1}")
        return 1 + m1
    dfs(root, 0)
    return ans

print("kết quả:", trace_dfs(build_nary(ex1)))

    nút 5: t=[] m1=0 m2=0 ans=0 -> trả 1
    nút 6: t=[] m1=0 m2=0 ans=0 -> trả 1
  nút 3: t=[1, 1] m1=1 m2=1 ans=2 -> trả 2
  nút 2: t=[] m1=0 m2=0 ans=2 -> trả 1
  nút 4: t=[] m1=0 m2=0 ans=2 -> trả 1
nút 1: t=[2, 1, 1] m1=2 m2=1 ans=3 -> trả 3
kết quả: 3


Trace Cách 2 (BFS hai lần) trên Ví dụ 3 — in khoảng cách theo tầng:

In [16]:
def trace_two_bfs(root):
    g = to_graph(root)
    def bfs(src):
        dist = {src: 0}
        q = deque([src])
        order = []
        while q:
            u = q.popleft()
            order.append((u.val, dist[u]))
            for v in g[u]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return order
    o1 = bfs(root)
    a_val, _ = o1[-1]
    print("BFS từ gốc  :", o1, "-> nút xa nhất", a_val)
    a = next(u for u in g if u.val == a_val)
    o2 = bfs(a)
    print(f"BFS từ {a_val:<5}:", o2, "-> đường kính", o2[-1][1])
    return o2[-1][1]

trace_two_bfs(build_nary(ex3))

BFS từ gốc  : [(1, 0), (2, 1), (3, 1), (4, 1), (5, 1), (6, 2), (7, 2), (8, 2), (9, 2), (10, 2), (11, 3), (12, 3), (13, 3), (14, 4)] -> nút xa nhất 14
BFS từ 14   : [(14, 0), (11, 1), (7, 2), (3, 3), (1, 4), (6, 4), (2, 5), (4, 5), (5, 5), (8, 6), (9, 6), (10, 6), (12, 7), (13, 7)] -> đường kính 7


7

## 9. Code nộp LeetCode

Bản chính: DFS hậu tố **lặp** (cùng công thức với doocs, không lo giới hạn đệ quy). Bản đệ quy của doocs là `SolutionDFS` ở trên.

In [17]:
class Solution:
    def diameter(self, root: 'Node') -> int:
        """
        :type root: 'Node'
        :rtype: int
        """
        if root is None:
            return 0
        order, stack = [], [root]
        while stack:
            u = stack.pop()
            order.append(u)
            stack.extend(u.children)
        h = {}
        ans = 0
        for u in reversed(order):
            m1 = m2 = 0
            for c in u.children:
                t = h[c]
                if t > m1:
                    m2, m1 = m1, t
                elif t > m2:
                    m2 = t
            ans = max(ans, m1 + m2)
            h[u] = 1 + m1
        return ans

for data, want in examples:
    assert Solution().diameter(build_nary(data)) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**DP top-2:** Mọi đường đi đơn `P` trong cây có duy nhất một nút `u` gần gốc nhất. `P` gồm tối đa hai đoạn đi xuống từ `u` vào **hai cây con khác nhau** của `u` (nếu cùng một cây con thì `u` không phải nút cao nhất). Độ dài mỗi đoạn ≤ `t(c) = 1 + chiều cao(c)` của con `c` tương ứng. Vậy `|P| ≤ m1 + m2` tại `u`, và `m1 + m2` đạt được (nối hai nhánh sâu nhất). Lấy max trên mọi `u` ⇒ đường kính. Giá trị trả về `1 + m1` đúng bằng số cạnh từ cha của `u` đi xuống sâu nhất qua `u`.

**BFS hai lần:** Bổ đề: với nút bất kỳ `s`, nút `a` xa `s` nhất là một đầu mút của một đường kính. Phác chứng minh: lấy đường kính `x–y`; nếu `a` không phải đầu mút của đường kính nào, xét đường từ `s` tới `x–y` và dùng bất đẳng thức khoảng cách trong cây (đường đi duy nhất) sẽ dựng được đường dài hơn `x–y` — mâu thuẫn. Khi đó BFS từ `a` cho khoảng cách xa nhất = đường kính. ∎

## 11. Phân tích Time / Space

`n ≤ 10⁴` số nút, `h ≤ 1000` độ sâu.

| Cách | Time | Space |
|---|---|---|
| 1. BFS từ mọi nút | `O(n²)` | `O(n)` |
| 2. BFS hai lần | `O(n)` | `O(n)` |
| 3. DFS đệ quy top-2 | `O(n)` | `O(h)` stack đệ quy |
| 4. DFS lặp top-2 | `O(n)` | `O(n)` |

Cách 3 tiết kiệm bộ nhớ nhất về lý thuyết nhưng phụ thuộc giới hạn đệ quy của Python.

## 12. Bộ test

Ví dụ đề, edge case (một nút, đường thẳng, hình sao, nhánh chỉ một con), random cây N-phân (seed cố định) so với brute force, và cây lớn `n = 10⁴` có độ sâu ~1000.

In [18]:
def random_nary(rng, n, max_depth=None):
    nodes = [Node(1)]
    depth = [0]
    for v in range(2, n + 1):
        while True:
            i = rng.randrange(len(nodes))
            if max_depth is None or depth[i] < max_depth:
                break
        child = Node(v)
        nodes[i].children.append(child)
        nodes.append(child)
        depth.append(depth[i] + 1)
    return nodes[0]

def chain(n):
    root = cur = Node(1)
    for v in range(2, n + 1):
        nxt = Node(v)
        cur.children.append(nxt)
        cur = nxt
    return root

METHODS = {
    "two_bfs": lambda r: SolutionTwoBFS().diameter(r),
    "dfs": lambda r: SolutionDFS().diameter(r),
    "iter": lambda r: Solution().diameter(r),
}
edge = [
    ([1], 0),
    ([1, None, 2], 1),
    ([1, None, 2, 3, 4, 5, 6], 2),
    ([1, None, 2, None, 3, None, 4, None, 5], 4),
    ([1, None, 2, 3, None, 4, None, 5, None, 6, None, 7], 6),
]
for data, want in examples + edge:
    r = build_nary(data)
    assert brute(r) == want, data
    for name, f in METHODS.items():
        assert f(r) == want, (name, data)
assert all(f(chain(50)) == 49 for f in METHODS.values())
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
rng = random.Random(1522)
for _ in range(400):
    r = random_nary(rng, rng.randint(1, 40))
    ref = brute(r)
    for name, f in METHODS.items():
        assert f(r) == ref, (name, serialize(r))
print("400 cây ngẫu nhiên: mọi cách khớp brute")

400 cây ngẫu nhiên: mọi cách khớp brute


In [20]:
big = random_nary(random.Random(7), 10**4)
deep = chain(1000)
for name in ["two_bfs", "iter"]:
    t = time.perf_counter()
    a, b = METHODS[name](big), METHODS[name](deep)
    print(f"{name:7s}: cây ngẫu nhiên {a}, chuỗi 1000 nút {b} ({time.perf_counter() - t:.3f}s)")
assert METHODS["two_bfs"](big) == METHODS["iter"](big) and METHODS["iter"](deep) == 999

two_bfs: cây ngẫu nhiên 34, chuỗi 1000 nút 999 (0.011s)
iter   : cây ngẫu nhiên 34, chuỗi 1000 nút 999 (0.003s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét đường đi qua gốc

Ví dụ 2: đường kính nằm ở cây con của nút 2, không qua gốc.

In [21]:
def wrong_root_only(root):
    def h(u):                                 # số cạnh nhánh sâu nhất từ u
        return 0 if not u.children else 1 + max(h(c) for c in u.children)
    ds = sorted((h(c) + 1 for c in root.children), reverse=True)
    return sum(ds[:2])                        # SAI: chỉ đường qua gốc

for data, want in examples:
    print(data[:6], "... | sai:", wrong_root_only(build_nary(data)), "| đúng:", want)

[1, None, 3, 2, 4, None] ... | sai: 3 | đúng: 3
[1, None, 2, None, 3, 4] ... | sai: 3 | đúng: 4
[1, None, 2, 3, 4, 5] ... | sai: 7 | đúng: 7


### Lỗi 2 — Cập nhật top-2 sai (quên đẩy `m1` xuống `m2`)

In [22]:
def wrong_top2(root):
    ans = 0
    def dfs(u):
        nonlocal ans
        m1 = m2 = 0
        for c in u.children:
            t = dfs(c)
            if t > m1:
                m1 = t                        # SAI: mất giá trị m1 cũ
            elif t > m2:
                m2 = t
        ans = max(ans, m1 + m2)
        return 1 + m1
    dfs(root)
    return ans

bad = build_nary([1, None, 2, 3, None, None, 4])   # con 2 nông, con 3 sâu hơn (thứ tự tăng)
print("sai:", wrong_top2(bad), "| đúng:", Solution().diameter(bad))

sai: 2 | đúng: 3


### Lỗi 3 — Đếm số **nút** thay vì số **cạnh**

Trộn hai quy ước: trả `1 + m1` (đếm nút) nhưng rồi cộng thêm 1 khi tính đường qua nút.

In [23]:
def wrong_nodes(root):
    ans = 0
    def dfs(u):
        nonlocal ans
        m1 = m2 = 0
        for c in u.children:
            t = dfs(c)
            if t > m1:
                m2, m1 = m1, t
            elif t > m2:
                m2 = t
        ans = max(ans, m1 + m2 + 1)           # SAI: +1 là đếm số nút trên đường
        return 1 + m1
    dfs(root)
    return ans

for data, want in examples:
    print("sai:", wrong_nodes(build_nary(data)), "| đúng:", want)

sai: 4 | đúng: 3
sai: 5 | đúng: 4
sai: 8 | đúng: 7


### Lỗi 4 — Đệ quy trên cây rất sâu

Đề giới hạn độ sâu ≤ 1000, rất sát giới hạn đệ quy mặc định của Python (stack của notebook cũng đã dùng vài chục tầng). Với cây sâu hơn (biến thể), đệ quy hỏng — ta bắt exception để in.

In [24]:
deep_tree = chain(3000)
try:
    print(SolutionDFS().diameter(deep_tree))
except RecursionError as e:
    print("sai: RecursionError ->", str(e)[:45])
print("đúng (lặp):", Solution().diameter(deep_tree))

sai: RecursionError -> maximum recursion depth exceeded
đúng (lặp): 2999


### Lỗi 5 — Lấy `max(chiều cao)` của hai nhánh **bất kỳ** chứ không phải hai nhánh **khác nhau**

Ví dụ dùng `2 * m1` (coi như đi xuống cùng một nhánh hai lần).

In [25]:
def wrong_double(root):
    ans = 0
    def dfs(u):
        nonlocal ans
        m1 = max((dfs(c) for c in u.children), default=0)
        ans = max(ans, 2 * m1)                # SAI: một nhánh dùng hai lần
        return 1 + m1
    dfs(root)
    return ans

for data, want in examples:
    print("sai:", wrong_double(build_nary(data)), "| đúng:", want)

sai: 4 | đúng: 3
sai: 6 | đúng: 4
sai: 8 | đúng: 7


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đường dài nhất giữa hai nút bất kỳ"** trên cây → DP hậu tố: hàm trả **một nhánh** cho cha, cập nhật đáp án toàn cục bằng **hai nhánh**.
- Pattern y hệt: LC 543 (Diameter of Binary Tree), LC 124 (Max Path Sum), LC 687 (Longest Univalue Path), LC 2246.
- Cây cho dưới dạng đồ thị → BFS hai lần là lựa chọn nhanh, không cần định gốc.
- Luôn chốt quy ước "cạnh hay nút" ngay từ đầu.
- Python + cây sâu → viết bản lặp (thứ tự tiền tố đảo ngược).

## 15. Biến thể của bài

1. **LC 543 – Diameter of Binary Tree:** chỉ 2 con → `m1 + m2 = h(left) + h(right)`.
2. **LC 1245 – Tree Diameter** (Premium): cây cho dưới dạng danh sách cạnh → BFS hai lần.
3. **Cây có trọng số cạnh:** thay "+1" bằng "+w"; BFS thay bằng DFS cộng trọng số (trọng số dương).
4. **LC 310 – Minimum Height Trees:** tâm cây nằm giữa đường kính.
5. **LC 124 – Binary Tree Maximum Path Sum:** giá trị nút có thể âm → `max(0, nhánh)`.
6. **Đường kính theo số nút** = đường kính theo cạnh + 1.

In [26]:
# Minh hoạ biến thể 3: cây N-phân có trọng số cạnh (lưu trong dict w[(cha.val, con.val)])
def weighted_diameter(root, w):
    order, stack = [], [root]
    while stack:
        u = stack.pop(); order.append(u); stack.extend(u.children)
    h, ans = {}, 0
    for u in reversed(order):
        m1 = m2 = 0
        for c in u.children:
            t = h[c] + w[(u.val, c.val)]
            if t > m1:
                m2, m1 = m1, t
            elif t > m2:
                m2 = t
        ans = max(ans, m1 + m2)
        h[u] = m1
    return ans

r = build_nary(ex1)
w = {(1, 3): 2, (1, 2): 10, (1, 4): 1, (3, 5): 4, (3, 6): 1}
print("đường kính có trọng số:", weighted_diameter(r, w), "(đường 5-3-1-2 = 4+2+10)")

đường kính có trọng số: 16 (đường 5-3-1-2 = 4+2+10)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi nút | Khoảng cách xa nhất từ mỗi nút | `O(n²)` | `O(n)` | Đối chiếu |
| 2. BFS hai lần | Nút xa nhất là đầu đường kính | `O(n)` | `O(n)` | Cây dạng đồ thị/danh sách cạnh |
| 3. DFS đệ quy top-2 | Hai nhánh sâu nhất tại mỗi đỉnh | `O(n)` | `O(h)` | Gọn, lời giải doocs |
| 4. DFS lặp top-2 | Như 3, duyệt tiền tố đảo ngược | `O(n)` | `O(n)` | An toàn với cây sâu |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Đường kính của `[1]`, của `[1, None, 2, 3, 4, 5, 6]` (hình sao) là bao nhiêu?

### Bài 2 (tính tay)
Với Ví dụ 3, chỉ ra một đường kính dài 7 (liệt kê giá trị các nút) và nút "đỉnh" của nó.

### Bài 3 (điền TODO)
Viết `diameter_path(root)`: trả **danh sách giá trị** các nút trên một đường kính.

### Bài 4 (tự viết lại)
Viết lại Cách 2 nhưng **không dựng đồ thị**: dùng dict `parent` để đi lên cha khi BFS.

### Bài 5 (mini challenge)
Viết `centers(root)`: danh sách giá trị các **tâm** của cây (nút có khoảng cách xa nhất tới mọi nút khác là nhỏ nhất), sắp tăng dần.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def diameter_path(root):
    # TODO: BFS hai lần, lần hai lưu cha của mỗi nút để dựng lại đường từ b về a
    pass

# Bài 4 — TODO
def two_bfs_parent(root):
    # TODO: parent[child] = node; hàng xóm của u = u.children + [parent[u]] nếu có
    pass

# Bài 5 — TODO
def centers(root):
    # TODO: lấy đường kính (Bài 3); tâm là 1 hoặc 2 nút ở giữa đường
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một nút không có cạnh. (2) Hình sao: hai lá bất kỳ đi qua gốc. (3) `0` và `2`.

**Bài 2.** (1) Tìm hai lá sâu nhất ở hai nhánh khác nhau của cùng một nút. (2) Nhánh sâu nhất của gốc: `1-3-7-11-14` (4 cạnh); nhánh sâu thứ hai qua con khác: `1-4-8-12` hoặc `1-5-9-13` (3 cạnh). (3) Ví dụ `14-11-7-3-1-5-9-13`, đỉnh là gốc 1 (đối chiếu với code ở mục 19).

**Bài 3.** (1) BFS từ gốc → `a`. (2) BFS từ `a`, lưu `prev`. (3) Từ nút xa nhất `b` đi theo `prev` về `a`.

**Bài 4.** (1) Duyệt một lượt để có `parent`. (2) Hàng xóm = con + cha. (3) Còn lại giống Cách 2.

**Bài 5.** (1) Mọi tâm nằm trên mọi đường kính. (2) Đường kính `L` cạnh, `L + 1` nút. (3) `L` chẵn → 1 tâm (nút giữa); `L` lẻ → 2 tâm.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
print("Bài 1:", Solution().diameter(build_nary([1])), Solution().diameter(build_nary([1, None, 2, 3, 4, 5, 6])))
assert Solution().diameter(build_nary([1])) == 0
assert Solution().diameter(build_nary([1, None, 2, 3, 4, 5, 6])) == 2

# Bài 3 (dùng luôn cho Bài 2)
def diameter_path_sol(root):
    g = to_graph(root)
    def bfs(src):
        prev = {src: None}
        q = deque([src])
        last = src
        while q:
            u = q.popleft()
            last = u
            for v in g[u]:
                if v not in prev:
                    prev[v] = u
                    q.append(v)
        return last, prev
    a, _ = bfs(root)
    b, prev = bfs(a)
    path = []
    while b is not None:
        path.append(b.val)
        b = prev[b]
    return path

p3 = diameter_path_sol(build_nary(ex3))
print("Bài 2: một đường kính của Ví dụ 3:", p3, "dài", len(p3) - 1)
assert len(p3) - 1 == 7

Bài 1: 0 2
Bài 2: một đường kính của Ví dụ 3: [13, 9, 5, 1, 3, 7, 11, 14] dài 7


In [29]:
def is_path(root, vals):
    g = to_graph(root)
    by = {u.val: u for u in g}
    return len(set(vals)) == len(vals) and all(by[vals[i + 1]] in g[by[vals[i]]] for i in range(len(vals) - 1))

# Bài 4
def two_bfs_parent_sol(root):
    parent = {root: None}
    stack = [root]
    while stack:
        u = stack.pop()
        for c in u.children:
            parent[c] = u
            stack.append(c)
    def nbrs(u):
        return u.children + ([parent[u]] if parent[u] is not None else [])
    def far(src):
        dist = {src: 0}
        q = deque([src]); last = src
        while q:
            u = q.popleft(); last = u
            for v in nbrs(u):
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
        return last, dist[last]
    a, _ = far(root)
    return far(a)[1]

rng = random.Random(4)
for _ in range(300):
    r = random_nary(rng, rng.randint(1, 30))
    d = brute(r)
    path = diameter_path_sol(r)
    assert len(path) - 1 == d and is_path(r, path)
    assert two_bfs_parent_sol(r) == d
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [30]:
# Bài 5
def centers_sol(root):
    path = diameter_path_sol(root)
    L = len(path) - 1
    if L % 2 == 0:
        return [path[L // 2]]
    return sorted([path[L // 2], path[L // 2 + 1]])

def brute_centers(root):
    g = to_graph(root)
    ecc = {}
    for s in g:
        dist = {s: 0}
        q = deque([s])
        while q:
            u = q.popleft()
            for v in g[u]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
        ecc[s.val] = max(dist.values())
    m = min(ecc.values())
    return sorted(v for v, e in ecc.items() if e == m)

print("Bài 5:", centers_sol(build_nary(ex1)), centers_sol(build_nary(ex2)), centers_sol(build_nary(ex3)))
for _ in range(300):
    r = random_nary(rng, rng.randint(1, 30))
    assert centers_sol(r) == brute_centers(r)
print("Đáp án Bài 1–5: mọi assert đều qua")

Bài 5: [1, 3] [2] [1, 3]
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đường dài nhất qua đỉnh" (hậu tố, top-2):**

```python
def dfs(u):                    # trả số cạnh nhánh sâu nhất + 1 (tức 1 + m1)
    nonlocal ans
    m1 = m2 = 0
    for c in u.children:
        t = dfs(c)
        if t > m1: m2, m1 = m1, t
        elif t > m2: m2 = t
    ans = max(ans, m1 + m2)    # hai nhánh khác nhau
    return 1 + m1              # chỉ một nhánh lên cha
```

**Checklist trước khi submit:**
- [ ] Đường kính **không bắt buộc qua gốc** — cập nhật `ans` tại mọi nút.
- [ ] Top-2: khi `t > m1` nhớ `m2 = m1` trước.
- [ ] Thống nhất đếm **cạnh** (cây 1 nút → 0).
- [ ] Cân nhắc bản lặp nếu cây sâu.
- [ ] Đúng chữ ký: `diameter(self, root: 'Node') -> int`.